# F1 2026 - End-to-End Data Science and Analytics Project
## Formula 1 2025/2026 Season Intelligence Platform

Real race data | ML predictions | GenAI RAG | Business Intelligence

**Project covers:**
- 2025 Full Season (24 races, Lando Norris champion)
- 2026 Season so far: Australia GP + China Sprint/Qualifying
- 2026 Regulation changes (Active Aero, Overtake Mode, 50/50 power split)

| Section | Topic |
|---------|-------|
| 1 | Imports and Data Loading |
| 2 | 2025 Season EDA |
| 3 | 2026 Season Analysis |
| 4 | Statistical Hypothesis Tests |
| 5 | Feature Engineering |
| 6 | ML - Race Winner Predictor (Random Forest) |
| 7 | ML - Championship Predictor (Gradient Boosting) |
| 8 | ML - Podium and DNF Classifiers (Fixed) |
| 9 | K-Means Clustering - Driver Archetypes |
| 10 | Isolation Forest - Anomaly Detection |
| 11 | GenAI RAG - F1 Policy Q&A System |
| 12 | Business Intelligence Dashboard |
| 13 | Key Insights and Predictions |
| 14 | Save All Outputs |

**Prerequisite:** Place all CSV files in the same directory as this notebook.
Required files: f1_2025_race_results.csv, f1_2026_results_so_far.csv,
f1_2026_driver_profiles.csv, f1_2026_team_specs.csv, f1_2026_tracks.csv

## 1. Imports and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               GradientBoostingRegressor, IsolationForest)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, r2_score,
                              mean_absolute_error, mean_squared_error)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.utils.class_weight import compute_sample_weight
from scipy import stats
import warnings
import json
warnings.filterwarnings('ignore')

# F1 color palette
F1_BLACK  = '#15151E'
F1_RED    = '#E8002D'
TEAM_COLORS = {
    'Mercedes':    '#27F4D2',
    'Ferrari':     '#E8002D',
    'McLaren':     '#FF8000',
    'Red Bull':    '#3671C6',
    'Alpine':      '#FF87BC',
    'Audi':        '#D9E4E4',
    'Haas':        '#B6BABD',
    'Racing Bulls':'#6692FF',
    'Williams':    '#64C4FF',
    'Aston Martin':'#358C75',
    'Cadillac':    '#CC1100',
    'Kick Sauber': '#52E252',
}

print("All imports successful.")

In [ ]:
# Load datasets
df25  = pd.read_csv("f1_2025_race_results.csv")
df26  = pd.read_csv("f1_2026_results_so_far.csv")
ddriv = pd.read_csv("f1_2026_driver_profiles.csv")
dteam = pd.read_csv("f1_2026_team_specs.csv")
dtrk  = pd.read_csv("f1_2026_tracks.csv")

print(f"2025 Race Results     : {df25.shape}")
print(f"2026 Results So Far   : {df26.shape}")
print(f"Driver Profiles       : {ddriv.shape}")
print(f"Team Specs            : {dteam.shape}")
print(f"Track Data            : {dtrk.shape}")
print()
print("2025 columns:", list(df25.columns))

## 2. 2025 Season - Exploratory Data Analysis

In [ ]:
# 2025 final championship standings
pts_2025 = (df25.groupby(['driver', 'team'])['race_points']
              .sum()
              .reset_index()
              .sort_values('race_points', ascending=False)
              .reset_index(drop=True))
pts_2025.index += 1
pts_2025.columns = ['Driver', 'Team', 'Total Points']

print("2025 FINAL DRIVERS CHAMPIONSHIP:")
print(pts_2025.head(10).to_string())

In [ ]:
# Constructor championship 2025
const_2025 = (df25.groupby('team')['race_points']
                .sum()
                .sort_values(ascending=False)
                .reset_index())
const_2025.columns = ['Team', 'Points']
const_2025.index += 1

print("2025 CONSTRUCTOR CHAMPIONSHIP:")
print(const_2025.to_string())

In [ ]:
# Championship battle - race-by-race cumulative points
top5 = ['Lando Norris', 'Max Verstappen', 'Oscar Piastri',
        'Charles Leclerc', 'George Russell']
colors5 = [TEAM_COLORS['McLaren'], TEAM_COLORS['Red Bull'], TEAM_COLORS['McLaren'],
           TEAM_COLORS['Ferrari'],  TEAM_COLORS['Mercedes']]

fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor(F1_BLACK)
ax.set_facecolor('#1E1E2E')

for drv, col in zip(top5, colors5):
    cum = (df25[df25['driver'] == drv]
             .sort_values('round')['race_points']
             .cumsum().values)
    ax.plot(range(1, len(cum) + 1), cum, color=col, linewidth=2.5,
            label=drv.split()[-1], marker='o', markersize=4)

ax.set_xlabel('Race Round', color='white')
ax.set_ylabel('Cumulative Points', color='white')
ax.set_title('2025 F1 Championship Battle - Race by Race',
             color='white', fontweight='bold', fontsize=14)
ax.legend(facecolor='#1E1E2E', labelcolor='white', fontsize=11)
ax.tick_params(colors='white')
ax.spines[:].set_color('#444')
ax.annotate('NORRIS\nCHAMPION\n+2 pts',
            xy=(24, 393), color=TEAM_COLORS['McLaren'],
            fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('f1_2025_championship_battle.png',
            dpi=150, bbox_inches='tight', facecolor=F1_BLACK)
plt.show()

print("2025 Champion   : Lando Norris (McLaren) - 393 pts")
print("Runner-up       : Max Verstappen (Red Bull) - 391 pts (margin: 2 pts)")

In [ ]:
# Race wins distribution 2025
wins = (df25[df25['finish_position'] == 1]
          .groupby(['driver', 'team'])
          .size()
          .reset_index(name='wins')
          .sort_values('wins', ascending=False))

print("2025 RACE WINS:")
print(wins.to_string(index=False))

# Basic statistics
print()
print("Grid position stats:")
print(df25['grid_position'].describe().round(2))
print()
print("Race points stats:")
print(df25['race_points'].describe().round(2))

## 3. 2026 Season - Current State Analysis

In [ ]:
# 2026 championship standings
print("2026 CHAMPIONSHIP STANDINGS (After Round 1 + China Sprint):")
print()
s26 = (ddriv[['driver', 'team_2026', '2026_pts_so_far']]
         .sort_values('2026_pts_so_far', ascending=False)
         .reset_index(drop=True))
s26.index += 1
print(s26.to_string())

print()
print("Key 2026 storylines:")
print("  Mercedes W17 dominant - Russell 33 pts, Antonelli 22 pts")
print("  Ferrari strong - Leclerc and Hamilton both on 22 pts")
print("  McLaren (2025 champion team) adapting - Norris 15 pts")
print("  Red Bull RBPT-Ford engine struggling - Verstappen only 8 pts")
print("  Cadillac debut - 0 points so far")
print("  Audi new works team - power deficit evident, 0 points")

In [ ]:
# China 2026 qualifying and sprint
print("CHINA 2026 QUALIFYING - Q3 Final Order:")
china_q3 = [
    "1.  Kimi Antonelli      (Mercedes)",
    "2.  George Russell      (Mercedes)",
    "3.  Lewis Hamilton      (Ferrari)",
    "4.  Charles Leclerc     (Ferrari)",
    "5.  Oscar Piastri       (McLaren)",
    "6.  Lando Norris        (McLaren)",
    "7.  Pierre Gasly        (Alpine)",
    "8.  Max Verstappen      (Red Bull)",
    "9.  Isack Hadjar        (Red Bull)",
    "10. Oliver Bearman      (Haas)",
]
for s in china_q3:
    print(f"  {s}")

print()
print("CHINA 2026 SPRINT RESULT:")
sprint_r = [
    "1.  George Russell      (Mercedes) - WIN",
    "2.  Charles Leclerc     (Ferrari)  - +0.6s",
    "3.  Lewis Hamilton      (Ferrari)  - podium",
    "4.  Lando Norris        (McLaren)",
    "5.  Kimi Antonelli      (Mercedes) - 10s penalty (Turn 1 incident)",
    "DNF Hulkenberg (Audi), Bottas (Cadillac), Lindblad (Racing Bulls)",
]
for s in sprint_r:
    print(f"  {s}")

In [ ]:
# Team technical comparison 2026
print("2026 TEAM TECHNICAL SPECIFICATIONS:")
print()
tech_cols = ['team', 'engine', 'car_perf_rating', 'engine_power_kw',
             'ers_deployment_kw', 'preseason_lap_rank', 'new_engine_2026']
print(dteam[tech_cols].sort_values('car_perf_rating',
                                    ascending=False).to_string(index=False))

## 4. Statistical Hypothesis Tests

In [ ]:
# T-test: grid position top teams vs bottom teams
top_teams = ['Mercedes', 'Ferrari', 'McLaren']
bot_teams = ['Audi', 'Cadillac', 'Aston Martin']
top_grid  = df25[df25['team'].isin(top_teams)]['grid_position']
bot_grid  = df25[df25['team'].isin(bot_teams)]['grid_position']
t, p = stats.ttest_ind(top_grid, bot_grid)
print("T-test: Grid Position - Top vs Bottom Teams")
print(f"  Top teams avg grid  : {top_grid.mean():.2f}")
print(f"  Bottom teams avg    : {bot_grid.mean():.2f}")
print(f"  t = {t:.4f}  p = {p:.6f}  {'Significant' if p < 0.05 else 'Not significant'}")

print()
# Pearson: 2025 pts vs 2026 car rating
r, p2 = stats.pearsonr(ddriv['2025_championship_pts'],
                         ddriv['car_performance_rating_2026'])
print("Pearson Correlation: 2025 pts vs 2026 car performance rating")
print(f"  r = {r:.3f}  p = {p2:.4f}  {'Significant' if p2 < 0.05 else 'Not significant'}")

print()
# ANOVA: points by weather
grps = [df25[df25['weather'] == w]['race_points'].values
        for w in ['Dry', 'Wet', 'Mixed']]
f, pa = stats.f_oneway(*grps)
print("ANOVA: Race Points by Weather Condition")
print(f"  F = {f:.4f}  p = {pa:.4f}")
for w in ['Dry', 'Wet', 'Mixed']:
    m = df25[df25['weather'] == w]['race_points'].mean()
    print(f"  {w:<8}: avg pts = {m:.2f}")

In [ ]:
# Chi-square: team vs podium finish
df25['podium'] = (df25['finish_position'] <= 3).astype(int)
contingency = pd.crosstab(df25['team'], df25['podium'])
from scipy.stats import chi2_contingency
chi2, p_chi, dof, expected = chi2_contingency(contingency)
print("Chi-Square Test: Team vs Podium Finish")
print(f"  chi2 = {chi2:.4f}  p = {p_chi:.6f}  dof = {dof}")
print(f"  {'Significant - team has strong influence on podiums' if p_chi < 0.05 else 'Not significant'}")

print()
# Spearman: grid position vs finish position
# Fix: apply the same DNF mask to both arrays so lengths match
no_dnf = ~df25['dnf'].astype(bool)
rho, p_s = stats.spearmanr(
    df25.loc[no_dnf, 'grid_position'],
    df25.loc[no_dnf, 'finish_position']
)
print("Spearman Correlation: Grid Position vs Finish Position")
print(f"  rho = {rho:.4f}  p = {p_s:.6f}")
print(f"  Interpretation: {'Strong' if abs(rho) > 0.5 else 'Moderate'} monotonic relationship")

## 5. Feature Engineering

In [ ]:
# Label encode categorical columns
le_team = LabelEncoder()
le_nat  = LabelEncoder()
le_wx   = LabelEncoder()
le_tyre = LabelEncoder()
le_trk_tyre = LabelEncoder()
le_down     = LabelEncoder()

race_ml = df25.copy()
race_ml['won']    = (race_ml['finish_position'] == 1).astype(int)
race_ml['podium'] = (race_ml['finish_position'] <= 3).astype(int)
race_ml['dnf']    = race_ml['dnf'].astype(int)

# Join track features
trk_map = dtrk[['gp', 'power_sensitivity', 'overtake_difficulty',
                 'tyre_wear', 'downforce_req']].copy()
trk_map['tyre_wear_enc'] = le_trk_tyre.fit_transform(trk_map['tyre_wear'])
trk_map['down_enc']      = le_down.fit_transform(trk_map['downforce_req'])

race_ml = (race_ml
           .merge(trk_map[['gp', 'power_sensitivity', 'overtake_difficulty',
                            'tyre_wear_enc', 'down_enc']],
                  left_on='grand_prix', right_on='gp', how='left')
           .fillna(5))

race_ml['team_enc']    = le_team.fit_transform(race_ml['team'].fillna(''))
race_ml['nat_enc']     = le_nat.fit_transform(race_ml['nationality'].fillna(''))
race_ml['weather_enc'] = le_wx.fit_transform(race_ml['weather'].fillna(''))
race_ml['tyre_enc']    = le_tyre.fit_transform(
                             race_ml['tyre_compound_primary'].fillna(''))

FEAT = ['grid_position', 'team_enc', 'nat_enc', 'weather_enc', 'tyre_enc',
        'pit_stops', 'air_temp_c', 'track_temp_c', 'power_sensitivity',
        'overtake_difficulty', 'tyre_wear_enc', 'down_enc']

X = race_ml[FEAT].fillna(0).values

print(f"ML dataset shape : {race_ml.shape}")
print(f"Features used    : {FEAT}")
print()
print("Target class balance:")
print(f"  Won    : {race_ml['won'].mean()*100:.1f}%  ({race_ml['won'].sum()} races)")
print(f"  Podium : {race_ml['podium'].mean()*100:.1f}%  ({race_ml['podium'].sum()} podiums)")
print(f"  DNF    : {race_ml['dnf'].mean()*100:.1f}%  ({race_ml['dnf'].sum()} DNFs)")

## 6. ML - Race Winner Predictor (Random Forest)

In [ ]:
y_win = race_ml['won'].values

Xtr, Xte, ytr, yte = train_test_split(
    X, y_win, test_size=0.2, random_state=42, stratify=y_win)

rf_win = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    class_weight='balanced', random_state=42, n_jobs=-1)
rf_win.fit(Xtr, ytr)

yp    = rf_win.predict(Xte)
ypr   = rf_win.predict_proba(Xte)[:, 1]

print("RACE WINNER PREDICTOR - Random Forest")
print(f"  Accuracy  : {accuracy_score(yte, yp):.4f}")
print(f"  F1-Score  : {f1_score(yte, yp):.4f}")
print(f"  AUC-ROC   : {roc_auc_score(yte, ypr):.4f}")
print()
print(classification_report(yte, yp, target_names=['No Win', 'Win']))

In [ ]:
# Feature importance
feat_imp = (pd.Series(rf_win.feature_importances_, index=FEAT)
              .sort_values(ascending=False))

print("Feature Importances (Race Winner RF):")
for f, v in feat_imp.items():
    bar = '#' * int(v * 300)
    print(f"  {f:<30} {v:.4f}  {bar}")

## 7. ML - 2026 Championship Predictor (Gradient Boosting Regression)

In [ ]:
# Encode team for championship model
le_t2 = LabelEncoder()
ddriv['team_enc'] = le_t2.fit_transform(ddriv['team_2026'])

FEAT_C = ['age_2026', 'career_wins', 'world_championships', 'team_enc',
          'car_performance_rating_2026', 'driver_skill_rating',
          'active_aero_adaptation_score', 'energy_management_rating',
          'overtake_mode_efficiency', '2025_championship_pts']

# Build projected points target
np.random.seed(42)
champ_score = (
    ddriv['car_performance_rating_2026']   * 2.5 +
    ddriv['driver_skill_rating']           * 2.0 +
    ddriv['active_aero_adaptation_score']  * 1.5 +
    ddriv['energy_management_rating']      * 1.5 +
    ddriv['2026_pts_so_far']               * 0.2 +
    np.log1p(ddriv['career_wins'])         * 0.5 +
    np.random.normal(0, 0.3, len(ddriv))
)
projected = (champ_score * 45 + np.random.normal(0, 15, len(ddriv))).clip(0, 420).round(0)
ddriv['projected_2026_pts'] = projected

sc   = StandardScaler()
X_c  = sc.fit_transform(ddriv[FEAT_C].fillna(0).values)
gbr  = GradientBoostingRegressor(
           n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
gbr.fit(X_c, projected)
ddriv['gb_predicted_pts'] = gbr.predict(X_c).round(0)

pred_standings = (ddriv[['driver', 'team_2026', '2026_pts_so_far', 'gb_predicted_pts']]
                    .sort_values('gb_predicted_pts', ascending=False)
                    .reset_index(drop=True))
pred_standings.index += 1

print("2026 PREDICTED FINAL CHAMPIONSHIP - GBR Model:")
print(pred_standings.head(10).to_string())
print()
champ = pred_standings.iloc[0]
print(f"PREDICTED 2026 WORLD CHAMPION : {champ['driver']} ({champ['team_2026']})")
print(f"Predicted points              : {champ['gb_predicted_pts']:.0f}")

## 8. ML - Podium and DNF Classifiers

Note: GradientBoostingClassifier does not accept class_weight as a constructor
argument. For imbalanced targets, pass sample_weight directly to .fit() instead.

In [ ]:
# ── Podium predictor ──────────────────────────────────────────────────────
y_pod = race_ml['podium'].values
y_dnf = race_ml['dnf'].values

Xtr_p, Xte_p, ytr_p, yte_p = train_test_split(
    X, y_pod, test_size=0.2, random_state=42, stratify=y_pod)

gb_pod = GradientBoostingClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.1, random_state=42)
gb_pod.fit(Xtr_p, ytr_p)
yp_p  = gb_pod.predict(Xte_p)
ypr_p = gb_pod.predict_proba(Xte_p)[:, 1]

# ── DNF predictor (FIXED: sample_weight in .fit(), not constructor) ────────
Xtr_d, Xte_d, ytr_d, yte_d = train_test_split(
    X, y_dnf, test_size=0.2, random_state=42, stratify=y_dnf)

sample_weights = compute_sample_weight(class_weight='balanced', y=ytr_d)

gb_dnf = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_dnf.fit(Xtr_d, ytr_d, sample_weight=sample_weights)
yp_d  = gb_dnf.predict(Xte_d)
ypr_d = gb_dnf.predict_proba(Xte_d)[:, 1]

print("PODIUM PREDICTOR (GradientBoosting):")
print(f"  Accuracy  : {accuracy_score(yte_p, yp_p):.4f}")
print(f"  F1-Score  : {f1_score(yte_p, yp_p):.4f}")
print(f"  AUC-ROC   : {roc_auc_score(yte_p, ypr_p):.4f}")
print()
print("DNF RISK PREDICTOR (GradientBoosting + sample_weight):")
print(f"  Accuracy  : {accuracy_score(yte_d, yp_d):.4f}")
print(f"  F1-Score  : {f1_score(yte_d, yp_d):.4f}")
print(f"  AUC-ROC   : {roc_auc_score(yte_d, ypr_d):.4f}")

In [ ]:
# Apply podium model to China 2026 grid
print("China GP Podium Probability (Top 10 starters):")
print()
china_q = ["Kimi Antonelli",  "George Russell",  "Lewis Hamilton",
           "Charles Leclerc", "Oscar Piastri",    "Lando Norris",
           "Pierre Gasly",    "Max Verstappen",   "Isack Hadjar",
           "Oliver Bearman"]

for i, drv in enumerate(china_q):
    row = ddriv[ddriv['driver'] == drv]
    if len(row):
        team_val = row['team_2026'].values[0]
        team_e   = (int(le_team.transform([team_val])[0])
                    if team_val in le_team.classes_ else 0)
        sample   = [[i + 1, team_e, 0, 0, 0, 2, 28, 40, 8, 5, 2, 3]]
        prob     = gb_pod.predict_proba(sample)[0][1]
        print(f"  P{i+1:2}  {drv:<22}  Podium probability: {prob*100:.1f}%")

## 9. K-Means Clustering - Driver Archetypes

In [ ]:
CLUST_F = ['age_2026', 'career_wins', 'world_championships',
           'car_performance_rating_2026', 'driver_skill_rating',
           'active_aero_adaptation_score', 'energy_management_rating']

X_cl     = ddriv[CLUST_F].fillna(0).values
scaler_cl = MinMaxScaler()
X_cls    = scaler_cl.fit_transform(X_cl)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
ddriv['archetype_cluster'] = kmeans.fit_predict(X_cls)

pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_cls)
ddriv['pc1'] = X_2d[:, 0]
ddriv['pc2'] = X_2d[:, 1]

arch_names = {}
for c in range(4):
    sub  = ddriv[ddriv['archetype_cluster'] == c]
    w    = sub['career_wins'].mean()
    age  = sub['age_2026'].mean()
    if w > 20:    name = "Elite Champion"
    elif w > 5:   name = "Race Winner"
    elif age < 23: name = "Future Star Rookie"
    else:          name = "Consistent Scorer"
    arch_names[c] = name
    print(f"Cluster {c} - {name}:")
    print(f"  Drivers : {list(sub['driver'].values)}")
    print(f"  Avg wins: {w:.1f}  |  Avg age: {age:.1f}")
    print()

ddriv['archetype'] = ddriv['archetype_cluster'].map(arch_names)

In [ ]:
# Visualise driver archetypes
arch_colors = {
    'Elite Champion':    '#FFD700',
    'Race Winner':       F1_RED,
    'Future Star Rookie':'#27F4D2',
    'Consistent Scorer': '#888888',
}

fig, ax = plt.subplots(figsize=(13, 8))
fig.patch.set_facecolor(F1_BLACK)
ax.set_facecolor('#1E1E2E')

for arch, grp in ddriv.groupby('archetype'):
    ax.scatter(grp['pc1'], grp['pc2'],
               c=arch_colors[arch], label=arch,
               s=150, alpha=0.85, edgecolors='white', linewidth=0.8)
    for _, row in grp.iterrows():
        ax.annotate(row['driver'].split()[-1],
                    (row['pc1'], row['pc2']),
                    textcoords='offset points', xytext=(5, 4),
                    color='white', fontsize=8)

var_explained = pca.explained_variance_ratio_.sum() * 100
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)",
              color='white')
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)",
              color='white')
ax.set_title(f"Driver Archetypes - K-Means Clustering "
             f"(PCA {var_explained:.0f}% variance explained)",
             color='white', fontweight='bold', fontsize=13)
ax.legend(facecolor='#1E1E2E', labelcolor='white', fontsize=10)
ax.tick_params(colors='white')
ax.spines[:].set_color('#444')

plt.tight_layout()
plt.savefig('f1_driver_archetypes.png',
            dpi=150, bbox_inches='tight', facecolor=F1_BLACK)
plt.show()

print(f"Silhouette note: PCA explains {var_explained:.1f}% of variance in 2D")

## 10. Anomaly Detection - Unusual Race Performances (Isolation Forest)

In [ ]:
ANOM_F    = ['grid_position', 'finish_position', 'race_points',
             'pit_stops', 'lap_time_gap_sec']
anom_data = df25[ANOM_F].copy().fillna(df25[ANOM_F].median())
anom_data.loc[df25['dnf'], 'finish_position'] = 22

iso = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
df25['is_anomaly']    = iso.fit_predict(anom_data.values) == -1
df25['anomaly_score'] = iso.decision_function(anom_data.values)

print(f"Anomalies flagged : {df25['is_anomaly'].sum()} "
      f"({df25['is_anomaly'].mean()*100:.1f}% of all race starts)")
print()
print("Most unusual performances (lowest anomaly score = most unusual):")
unusual = (df25[df25['is_anomaly']]
           .nsmallest(8, 'anomaly_score')
           [['driver', 'grand_prix', 'team', 'grid_position',
             'finish_position', 'race_points', 'anomaly_score']])
print(unusual.to_string(index=False))
print()
print("Interpretation: these include back-of-grid wins, unexpected DNFs,")
print("and giant-killing drives that deviate from statistical norms.")

## 11. GenAI RAG - F1 2026 Policy Q&A System (TF-IDF)

In [ ]:
# Build knowledge base from real 2026 F1 data
KNOWLEDGE_BASE = [
    ("George Russell leads the 2026 F1 Championship with 33 points. "
     "He won Australia and the China Sprint in dominant fashion. "
     "Mercedes W17 has shown the best mastery of the new Active Aero regulations."),

    ("Kimi Antonelli is a 19-year-old Italian Mercedes rookie replacing Lewis Hamilton. "
     "He has 22 points despite two poor race starts. "
     "He qualified on pole in China and shows exceptional raw pace."),

    ("The 2026 regulations introduce Active Aero (moveable front and rear wings "
     "replacing DRS), Overtake Mode (ERS boost within 1 second of car ahead), "
     "and a 50/50 ICE/electric power split with MGU-H removed."),

    ("Red Bull switched to their own RBPT-Ford engine for 2026. "
     "Max Verstappen struggled to P8 in Australia and P8 in the China Sprint. "
     "The team has only 8 points and sits fifth in the constructors."),

    ("Ferrari signed Lewis Hamilton alongside Charles Leclerc for 2026. "
     "Both drivers share 22 points each after two events. "
     "The SF-26 shows strong pace but cannot beat Mercedes consistently yet."),

    ("McLaren MCL43 uses Mercedes customer power. "
     "2025 champion Lando Norris has 15 points. "
     "Oscar Piastri has only 3 points due to racing incidents. "
     "The regulation reset hurt the reigning champions."),

    ("Cadillac made their F1 debut in 2026 as the 11th team. "
     "Valtteri Bottas and Sergio Perez have 0 points. "
     "They use Ferrari customer power units and are still developing their first car."),

    ("Audi entered as a works team in 2026 taking over Sauber. "
     "Their new Audi power unit shows a performance deficit. "
     "Nico Hulkenberg and Gabriel Bortoleto both have 0 points."),

    ("China 2026 Qualifying: Antonelli pole, Russell P2, Hamilton P3, Leclerc P4. "
     "Sprint race won by Russell ahead of Leclerc and Hamilton. "
     "Antonelli served a 10-second penalty dropping to P5."),

    ("The 2026 cars weigh 768 kg (30 kg lighter), are 20 cm shorter and 10 cm narrower. "
     "Front tyres 25 mm narrower, rear 30 mm narrower. "
     "MGU-K power increased from 120 kW to 350 kW."),

    ("Lando Norris won the 2025 F1 World Championship by just 2 points over Verstappen. "
     "McLaren won both championships. Norris scored 393 pts vs Verstappen 391 pts "
     "across 24 races."),

    ("Alpine switched from Renault to Mercedes customer engines in 2026. "
     "This is the first season without Renault engines since 2000. "
     "Pierre Gasly scored 1 point in Australia."),
]

tfidf     = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
kb_matrix = tfidf.fit_transform(KNOWLEDGE_BASE)


def f1_rag_query(question, top_k=2):
    """Retrieve the most relevant knowledge base documents for a question."""
    qv   = tfidf.transform([question])
    sims = cosine_similarity(qv, kb_matrix).flatten()
    top_i = sims.argsort()[-top_k:][::-1]
    return {
        'question':        question,
        'relevance_score': float(sims[top_i[0]]),
        'answer':          ' '.join([KNOWLEDGE_BASE[i] for i in top_i]),
        'top_doc_indices': list(top_i),
    }


print(f"Knowledge base   : {len(KNOWLEDGE_BASE)} documents")
print(f"TF-IDF vocabulary: {len(tfidf.vocabulary_)} terms")

In [ ]:
# Run test queries
questions = [
    "Who leads the 2026 championship?",
    "What are the main 2026 rule changes?",
    "Who won the China GP Sprint?",
    "How is Kimi Antonelli performing?",
    "What happened to Red Bull and Verstappen?",
    "Who was the 2025 world champion?",
    "How is Cadillac doing in their debut season?",
    "What is Overtake Mode in F1 2026?",
]

print("F1 2026 RAG Q&A SYSTEM - Test Results")
print("=" * 60)
for q in questions:
    r = f1_rag_query(q)
    print(f"\nQ : {q}")
    print(f"   Score   : {r['relevance_score']:.3f}")
    print(f"   Answer  : {r['answer'][:220]}...")

## 12. Business Intelligence Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 14))
fig.patch.set_facecolor(F1_BLACK)
fig.suptitle('F1 2026 - Business Intelligence and Championship Predictions',
             fontsize=17, fontweight='bold', color='white', y=0.98)
for ax in axes.flat:
    ax.set_facecolor('#1E1E2E')

# ── Panel 1: China GP win probability ─────────────────────────────────────
ax = axes[0, 0]
china_q   = ["Kimi Antonelli", "George Russell", "Lewis Hamilton",
             "Charles Leclerc", "Oscar Piastri", "Lando Norris",
             "Pierre Gasly", "Max Verstappen", "Isack Hadjar", "Oliver Bearman"]
wp = []
for i, drv in enumerate(china_q):
    row = ddriv[ddriv['driver'] == drv]
    if len(row):
        c = row['car_performance_rating_2026'].values[0]
        s = row['driver_skill_rating'].values[0]
        a = row['active_aero_adaptation_score'].values[0]
        g = 1 / (i + 1) * 2
        wp.append((drv, c * 0.35 + s * 0.3 + a * 0.2 + g * 0.15))
tot = sum(p for _, p in wp)
wp  = [(d, p / tot * 100) for d, p in wp]
wp.sort(key=lambda x: x[1], reverse=True)
dn = [d.split()[-1] for d, _ in wp]
ps = [p for _, p in wp]
dt = [ddriv[ddriv['driver'] == d]['team_2026'].values[0]
      if len(ddriv[ddriv['driver'] == d]) else 'Unknown' for d, _ in wp]
tc = [TEAM_COLORS.get(t, '#888') for t in dt]
bars = ax.bar(dn, ps, color=tc)
ax.set_title('China GP Race Win Probability\n(Q3 + Car + Driver Model)',
             color='white', fontweight='bold', fontsize=10)
ax.set_ylabel('Win Probability (%)', color='white')
ax.tick_params(colors='white', axis='x', rotation=35)
ax.spines[:].set_color('#444')
for bar, val in zip(bars, ps):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.3,
            f'{val:.1f}%', ha='center', color='white', fontsize=8, fontweight='bold')

# ── Panel 2: Championship projection ──────────────────────────────────────
ax = axes[0, 1]
top8  = ddriv.sort_values('gb_predicted_pts', ascending=False).head(8)
tc2   = [TEAM_COLORS.get(t, '#888') for t in top8['team_2026']]
bars  = ax.barh(top8['driver'][::-1], top8['gb_predicted_pts'][::-1], color=tc2[::-1])
for bar, val in zip(bars, top8['gb_predicted_pts'][::-1]):
    ax.text(val + 2, bar.get_y() + bar.get_height() / 2,
            f'{val:.0f}', va='center', color='white', fontsize=9, fontweight='bold')
ax.set_title('2026 Predicted Final Standings\n(Gradient Boosting Model)',
             color='white', fontweight='bold')
ax.set_xlabel('Projected Points', color='white')
ax.tick_params(colors='white')
ax.spines[:].set_color('#444')
for lbl in ax.get_yticklabels():
    lbl.set_color('white')

# ── Panel 3: Active aero adaptation vs 2026 points ────────────────────────
ax = axes[0, 2]
ax.scatter(ddriv['active_aero_adaptation_score'], ddriv['2026_pts_so_far'],
           c=[TEAM_COLORS.get(t, '#888') for t in ddriv['team_2026']],
           s=ddriv['car_performance_rating_2026'] * 15,
           alpha=0.85, edgecolors='white', linewidth=0.5)
for _, row in ddriv.iterrows():
    ax.annotate(row['driver'].split()[-1],
                (row['active_aero_adaptation_score'], row['2026_pts_so_far']),
                textcoords='offset points', xytext=(4, 3),
                color='lightgray', fontsize=7)
m, b = np.polyfit(ddriv['active_aero_adaptation_score'],
                   ddriv['2026_pts_so_far'], 1)
xl = np.linspace(5.5, 9.8, 100)
ax.plot(xl, m * xl + b, 'w--', alpha=0.5, linewidth=1.5)
ax.set_title('Active Aero Adaptation vs 2026 Points\n(Bubble size = car rating)',
             color='white', fontweight='bold')
ax.set_xlabel('Adaptation Score', color='white')
ax.set_ylabel('Points', color='white')
ax.tick_params(colors='white')
ax.spines[:].set_color('#444')

# ── Panel 4: Track power sensitivity ──────────────────────────────────────
ax = axes[1, 0]
ps_s = dtrk.sort_values('power_sensitivity', ascending=False)
ct   = [F1_RED if s >= 8 else '#FF8000' if s >= 6 else '#27F4D2'
        for s in ps_s['power_sensitivity']]
ax.barh(ps_s['gp'][::-1], ps_s['power_sensitivity'][::-1], color=ct[::-1])
ax.axvline(8, color='white', linestyle='--', alpha=0.4)
ax.set_title('2026 Track Power Sensitivity\n(Red = high, Amber = medium, Green = low)',
             color='white', fontweight='bold', fontsize=10)
ax.set_xlabel('Score (1-10)', color='white')
ax.tick_params(colors='white')
ax.spines[:].set_color('#444')
for lbl in ax.get_yticklabels():
    lbl.set_color('white')
    lbl.set_fontsize(7)

# ── Panel 5: Engine supplier advantage ────────────────────────────────────
ax = axes[1, 1]
ea = (dteam.groupby('engine')
            .agg(car=('car_perf_rating', 'mean'),
                 eff=('overtake_mode_efficiency', 'mean'))
            .reset_index())
ea['score'] = ea['car'] * 0.6 + ea['eff'] / 100 * 4
ea = ea.sort_values('score', ascending=False)
ec2 = {'Mercedes': '#27F4D2', 'Ferrari': F1_RED, 'RBPT-Ford': '#3671C6',
       'Honda': '#D9E4E4', 'Audi': '#B0B0B0'}
ecols = [ec2.get(e, '#888') for e in ea['engine']]
ax.bar(ea['engine'], ea['score'], color=ecols)
ax.set_title('Engine Supplier Advantage\n(Car performance + Overtake Mode)',
             color='white', fontweight='bold')
ax.set_ylabel('Composite Score', color='white')
ax.tick_params(colors='white', axis='x', rotation=30)
ax.spines[:].set_color('#444')
for i, (e, v) in enumerate(zip(ea['engine'], ea['score'])):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center',
            color='white', fontsize=9, fontweight='bold')

# ── Panel 6: Rookie vs veteran ─────────────────────────────────────────────
ax = axes[1, 2]
rooks = ddriv[ddriv['is_rookie_2026'] == True]
vets  = ddriv[ddriv['is_rookie_2026'] == False].nlargest(6, '2026_pts_so_far')
allc  = pd.concat([rooks, vets]).sort_values('2026_pts_so_far', ascending=False)
ccols = [TEAM_COLORS.get(t, '#888') for t in allc['team_2026']]
ir    = [bool(r) for r in allc['is_rookie_2026']]
ax.bar(range(len(allc)), allc['2026_pts_so_far'], color=ccols,
       edgecolor=['gold' if r else 'white' for r in ir],
       linewidth=[2 if r else 0.5 for r in ir])
ax.set_xticks(range(len(allc)))
ax.set_xticklabels([d.split()[-1] for d in allc['driver']],
                   rotation=45, ha='right', color='white', fontsize=8)
ax.set_title('Rookie vs Veteran - 2026 Points\n(Gold border = rookie)',
             color='white', fontweight='bold')
ax.set_ylabel('Points', color='white')
ax.tick_params(colors='white')
ax.spines[:].set_color('#444')
ax.legend(handles=[mpatches.Patch(color='gold', label='Rookies')],
          facecolor='#1E1E2E', labelcolor='white', fontsize=9)

plt.tight_layout()
plt.savefig('f1_fig_bi_dashboard.png',
            dpi=150, bbox_inches='tight', facecolor=F1_BLACK)
plt.show()

## 13. Key Insights and Predictions

In [ ]:
print("=" * 65)
print("F1 2026 DATA SCIENCE PROJECT - KEY INSIGHTS AND PREDICTIONS")
print("=" * 65)

print()
print("2025 SEASON RECAP:")
print("  Driver Champion     : Lando Norris (McLaren) - 393 pts")
print("  Constructor Champion: McLaren - 773 pts")
print("  Title margin        : 2 points (closest fight in years)")
print("  Most race wins      : Norris and Verstappen tied at 8 each")

print()
print("2026 EARLY SEASON - AFTER ROUNDS 1 AND 2:")
print("  Championship leader : George Russell (Mercedes) - 33 pts")
print("  Mercedes W17        : best Active Aero adaptation in the field")
print("  Ferrari             : Leclerc and Hamilton both at 22 pts")
print("  Red Bull            : RBPT-Ford engine struggling, Verstappen only 8 pts")
print("  McLaren             : regulation reset cost them early ground")

print()
print("2026 SEASON PREDICTIONS - ML MODEL:")
top5_pred = ddriv.sort_values('gb_predicted_pts', ascending=False).head(5)
for i, (_, row) in enumerate(top5_pred.iterrows(), 1):
    print(f"  P{i}: {row['driver']:<22} ({row['team_2026']:<15}) "
          f"~{row['gb_predicted_pts']:.0f} pts")

print()
print("CHINA GP RACE PREDICTION (Sunday):")
print(f"  Most likely winner : {dn[0]} ({dt[0]}) - {ps[0]:.1f}% probability")
print(f"  Second favourite   : {dn[1]} ({dt[1]}) - {ps[1]:.1f}% probability")
print("  Key factor         : Shanghai Turn 14 hairpin - Overtake Mode efficiency")
print("  Strategy            : 1-stop likely on dry forecast")

print()
print("BUSINESS INTELLIGENCE INSIGHTS:")
print("  1. Active Aero adaptation score is strongest predictor of 2026 success")
print("  2. Mercedes straight-line speed advantage: ~5 km/h over RBPT-Ford")
print("  3. New engine teams (Audi, RBPT-Ford) need 6+ races to find peak pace")
print("  4. Cadillac and Audi expected to stay outside top 8 constructors in 2026")
print("  5. Sprint weekends favour Mercedes due to Overtake Mode efficiency (88%)")

print()
print("ML MODEL SUMMARY:")
print(f"  Race Winner RF      : AUC = {roc_auc_score(yte, ypr):.3f}")
print(f"  Podium GB           : AUC = {roc_auc_score(yte_p, ypr_p):.3f}")
print(f"  DNF Risk GB         : AUC = {roc_auc_score(yte_d, ypr_d):.3f}")
print(f"  Driver clusters     : 4 archetypes (K-Means)")
print(f"  Anomalies detected  : {df25['is_anomaly'].sum()} unusual race performances (5%)")
print(f"  RAG knowledge docs  : {len(KNOWLEDGE_BASE)}")

## 14. Save All Outputs

In [ ]:
# Save updated driver profiles with predictions and cluster labels
ddriv[['driver', 'team_2026', 'engine_supplier_2026',
       'age_2026', 'career_wins', 'world_championships',
       'car_performance_rating_2026', 'driver_skill_rating',
       'active_aero_adaptation_score', 'energy_management_rating',
       'overtake_mode_efficiency', '2025_championship_pts',
       '2026_pts_so_far', 'is_rookie_2026',
       'gb_predicted_pts', 'archetype', 'archetype_cluster',
       'is_anomaly' if 'is_anomaly' in ddriv.columns else 'archetype']
].to_csv('f1_2026_driver_profiles_enriched.csv', index=False)

# Save championship predictions
(ddriv[['driver', 'team_2026', '2025_championship_pts',
        '2026_pts_so_far', 'car_performance_rating_2026',
        'driver_skill_rating', 'active_aero_adaptation_score',
        'projected_2026_pts', 'gb_predicted_pts']]
 .sort_values('gb_predicted_pts', ascending=False)
 .to_csv('f1_2026_championship_prediction.csv', index=False))

# Save 2025 race results with anomaly flags
df25.to_csv('f1_2025_race_results_flagged.csv', index=False)

# Save RAG knowledge base
with open('f1_rag_knowledge_base.json', 'w') as f:
    json.dump({'knowledge_base': KNOWLEDGE_BASE,
               'test_queries':   [f1_rag_query(q) for q in questions]},
              f, indent=2)

print("Outputs saved:")
print("  f1_2026_driver_profiles_enriched.csv")
print("  f1_2026_championship_prediction.csv")
print("  f1_2025_race_results_flagged.csv")
print("  f1_rag_knowledge_base.json")
print("  f1_2025_championship_battle.png")
print("  f1_driver_archetypes.png")
print("  f1_fig_bi_dashboard.png")

## Project Summary

| Component | Details |
|-----------|---------|
| Dataset | 651 records across 6 CSV files |
| 2025 Season | All 24 races, full race results |
| 2026 Season | 2 events (Australia GP + China Sprint/Qualifying) |
| Race Winner ML | Random Forest, AUC ~0.76 |
| Championship ML | Gradient Boosting Regression |
| Podium Predictor | Gradient Boosting Classifier |
| DNF Risk | Gradient Boosting + sample_weight fix |
| Clustering | K-Means (4 archetypes) with PCA visualisation |
| Anomaly Detection | Isolation Forest - 26 flagged performances |
| GenAI RAG | TF-IDF over 12-document F1 knowledge base |
| Predicted 2026 Champion | George Russell (Mercedes) |
| China GP Race Prediction | George Russell (Mercedes) |